<a href="https://colab.research.google.com/github/Claudiopj88/Atividades_ANN/blob/main/Atividade_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz
!rm -r aclImdb/train/unsup

In [ ]:
import os, pathlib, shutil, random
base_dir = pathlib.Path("aclImdb")
train_dir = base_dir / "train"
val_dir = base_dir / "val"
test_dir = base_dir / "test"
for category in ["pos", "neg"]:
  os.makedirs(val_dir / category)
  files = os.listdir(train_dir / category)
  random.shuffle(files)
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname, val_dir / category / fname)

In [ ]:
from tensorflow import keras

batch_size = 32
train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size)

In [ ]:
from tensorflow.keras.layers import TextVectorization
max_length = 600
max_tokens=20000
text_vectorization = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length)
text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)
int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)

In [ ]:
!pip install transformers

In [ ]:
from transformers import pipeline

pipe = pipeline("text-classification", model="lvwerra/distilbert-imdb")

In [ ]:
results = pipe(["This is a great movie", "This is a bad movie"])
for result in results:
  print(f"label: {result['label']}, with score: {round(result['score'], 4)}")

In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np
import time

y_pred = []
test_labels = []
start = time.time()
# Iterate over the original test_ds which yields (text_string_tensor, label_tensor)
for batch, (texts, targets) in enumerate(test_ds):
  # Process each individual text and its corresponding target label in the batch
  for i in range(len(texts)):
    # Extract the text string and decode it from bytes to utf-8 string
    current_text = texts[i].numpy().decode('utf-8')
    # Pass the string to the pipeline, truncating if necessary
    yp = pipe(current_text, truncation=True, max_length=512) # pipe expects a string or list of strings

    # Get the true label: targets[i] is a scalar tensor. Convert to int, 1 for positive, 0 for negative
    # Assuming `1` corresponds to positive class in `test_ds` (e.g., from text_dataset_from_directory)
    true_label = int(targets[i].numpy() == 1)
    test_labels.append(true_label)

    # Get the predicted label from the pipeline output
    # 'lvwerra/distilbert-imdb' outputs 'POSITIVE' or 'NEGATIVE'
    predicted_label_is_positive = (yp[0]['label'] == 'POSITIVE')
    y_pred.append(predicted_label_is_positive)

  print(f"{batch} {time.time()-start:.2f} s - Accuracy: {accuracy_score(test_labels, y_pred):.4f}")
print("Final Accuracy", accuracy_score(test_labels, y_pred))